In [16]:
import pandas as pd
import geopandas as gpd
import pdfplumber

from pathlib import Path

In [17]:
with pdfplumber.open(Path.cwd().parent.parent.joinpath("raw_data", "Capital Cost and Performance Characteristics for Utility-Scale Electric Power Generating Technologies.pdf")) as pdf:
    location_adjustment_page =  pdf.pages[170] # page 171 of pdf
    table_data = location_adjustment_page.extract_table()
    df_cost = pd.DataFrame(table_data[1:], columns=table_data[0]) # first row set as column names, else is data

In [18]:
df_cost.head()

,State,City,Base Project Cost ($/kW ),Location Variation,Delta Cost Difference ($/kW),Total Location Project Cost ($/kW)
0,Alabama,Huntsville,"8,936",0.99,(76),8860
1,Arizona,Phoenix,"8,936",1.01,60,8996
2,Arkansas,Little Rock,"8,936",1.00,26,8962
3,California,Bakersfield,"8,936",1.13,"1,193",10129
4,California,Los Angeles,"8,936",1.15,"1,305",10241


In [19]:
df_cost["Location Variation"] = df_cost["Location Variation"].astype(float)

In [20]:
# state and location variation from pdf
df_cost = df_cost.groupby('State').agg("mean", numeric_only=True).reset_index()
df_cost.head()

,State,Location Variation
0,Alabama,0.99
1,Arizona,1.01
2,Arkansas,1.00
3,California,1.16
4,Colorado,0.98


In [21]:
# geo id and counties from candidates_ranked
data = Path.cwd().parent.parent.joinpath("processed_data", "candidates_ranked.csv")
df = pd.read_csv(data)
df.head()

,geo_id,county_name,population,median_household_income,housing_units,total_energy_consumption_mwh,data_centers_count,sfha_area,pct_sfha,lake_count,...,max_voltage,average_voltage,protected_count,total_protected_area_m,pct_protected,county_area_km2,population_density,has_plant,mcda_score,rank
0,40075,Kiowa County,8181.0,42679.0,4700.0,157935.0,0.0,2.112674e+08,0.079146,10,...,138.0,110.400000,2.0,4.108833e+07,0.015393,2669.336760,3.064806,0,0.529671,1880.0
1,46079,Lake County,10993.0,74884.0,5714.0,170505.0,0.0,1.890356e+08,0.126898,48,...,69.0,69.000000,156.0,1.136746e+08,0.076309,1489.662955,7.379522,0,0.591694,1349.0
2,37033,Caswell County,22563.0,56999.0,10493.0,295896.0,0.0,6.618573e+07,0.059609,4,...,230.0,230.000000,16.0,1.409890e+07,0.012698,1110.336635,20.320864,0,0.637095,799.0
3,48377,Presidio County,5433.0,29012.0,3396.0,106309.0,0.0,0.000000e+00,0.000000,0,...,69.0,69.000000,16.0,3.486332e+08,0.034909,9986.831983,0.544016,0,0.585314,1404.0
4,39057,Greene County,174322.0,81243.0,71471.0,2239244.0,0.0,1.002618e+08,0.092999,8,...,345.0,112.909091,58.0,1.130754e+07,0.010488,1078.100699,161.693616,0,0.596162,1284.0


In [22]:
# state fips and geo id from county shp
county_path = Path.cwd().parent.parent.joinpath('raw_data', 'county_boundaries_2025', 'tl_2025_us_county.shp')
df_county = gpd.read_file(county_path)
df_county = df_county[['STATEFP','GEOID','NAMELSAD']]
df_county = df_county.rename(columns={'STATEFP':'state_fips','GEOID':'geo_id','NAMELSAD':'county_name'})
df_county.head(2)
df_county['geo_id'] = df_county['geo_id'].astype(int)

In [23]:
df = df.merge(df_county[['state_fips','geo_id']],left_on='geo_id',right_on='geo_id',how='left')
df = df[['geo_id','county_name','state_fips']]
df['state_fips'] = df['state_fips'].astype(int)
df.head()

,geo_id,county_name,state_fips
0,40075,Kiowa County,40
1,46079,Lake County,46
2,37033,Caswell County,37
3,48377,Presidio County,48
4,39057,Greene County,39


In [24]:
fips_to_state = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas',
    6: 'California', 8: 'Colorado', 9: 'Connecticut', 10: 'Delaware',
    11: 'District of Columbia', 12: 'Florida', 13: 'Georgia', 16: 'Idaho',
    17: 'Illinois', 18: 'Indiana', 19: 'Iowa', 20: 'Kansas',
    21: 'Kentucky', 22: 'Louisiana', 23: 'Maine', 24: 'Maryland',
    25: 'Massachusetts', 26: 'Michigan', 27: 'Minnesota', 28: 'Mississippi',
    29: 'Missouri', 30: 'Montana', 31: 'Nebraska', 32: 'Nevada',
    33: 'New Hampshire', 34: 'New Jersey', 35: 'New Mexico', 36: 'New York',
    37: 'North Carolina', 38: 'North Dakota', 39: 'Ohio', 40: 'Oklahoma',
    41: 'Oregon', 42: 'Pennsylvania', 44: 'Rhode Island', 45: 'South Carolina',
    46: 'South Dakota', 47: 'Tennessee', 48: 'Texas', 49: 'Utah',
    50: 'Vermont', 51: 'Virginia', 53: 'Washington', 54: 'West Virginia',
    55: 'Wisconsin', 56: 'Wyoming'
}

df['State'] = df['state_fips'].map(fips_to_state)

# merge
df_merge = df.merge(df_cost[['State', 'Location Variation']],
                               left_on='State',
                               right_on='State',
                               how='left').drop(columns='State')

df_merge = df_merge.rename(columns={'Location Variation': 'location_factor'})

print('Missing:', df_merge['location_factor'].isna().sum())
df_merge.head()

Missing: 15


,geo_id,county_name,state_fips,location_factor
0,40075,Kiowa County,40,0.97
1,46079,Lake County,46,0.99
2,37033,Caswell County,37,0.99
3,48377,Presidio County,48,0.95
4,39057,Greene County,39,0.99


In [25]:
df_merge['geo_id'] = df_merge['geo_id'].astype(str).str.zfill(5)

In [26]:
df_merge[df_merge['location_factor'].isna()]
# Alaska, Hawaii, and the territories don't have a location factor

,geo_id,county_name,state_fips,location_factor
155,78030,St. Thomas Island,78,NaN
187,02198,Prince of Wales-Hyder Census Area,2,NaN
226,60040,Swains Island,60,NaN
297,66010,Guam,66,NaN
301,02110,Juneau City and Borough,2,NaN
325,69100,Rota Municipality,69,NaN
846,69110,Saipan Municipality,69,NaN
980,60020,Manu'a District,60,NaN
981,69120,Tinian Municipality,69,NaN
1004,72097,Mayagüez Municipio,72,NaN


In [27]:
df_merge = df_merge[['geo_id','county_name','location_factor']]

In [28]:
# saving as csv to 'processed_data' folder
output_path = Path.cwd().parent.parent.joinpath('processed_data', 'location_factors.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)
df_merge.to_csv(output_path, index=False)